## Preprocess without HVG selection

HVG selection, normalization, log1p, standardization performed during cross validation in the BINN pipeline.

In [1]:
import preprocess as pre
from pathlib import Path
import pandas as pd
import importlib.util
import anndata as ad
import scanpy as sc

In [2]:
# Determine parent folder
base_dir = Path.cwd().parent
path = base_dir / "pipeline_paths_mnt.py"

# Import module dynamically
spec = importlib.util.spec_from_file_location("ppaths", path)
ppaths = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ppaths)

In [3]:
# VARIABLES
# Quality control
min_cells = 200 
min_genes = 200
mito_threshold: float = 5.0
min_cells_per_subject: int = 30

# train-test split
test_size: float = 0.2
random_state: int = 17

# to stratify AD+NCI v MCIP
stratify_key: str = 'label'

In [4]:
# sets variables in file and folder names
run_vars = f'mg_{min_genes}_mc_{min_cells}' 

# to keep consistent paths
pp = ppaths.PipelinePaths("dev_local", pseudo_non_hvg_run_vars=run_vars)

base path found (D:\BBTX11-VT26-07\simulated): True
data path found (D:\BBTX11-VT26-07\simulated\data): True


In [5]:
in_path = Path(pp.conv_data_path)

# datasets to include (keys are readable labels)
datasets = {}

COMBINED = False
if COMBINED:
    cell_types = ['Inh', 'Exc']
    combined_file = in_path / f'{'_'.join(cell_types)}_combined.h5ad'

    if combined_file.exists():
        adata_combined = ad.read_h5ad(combined_file)
    else:
        input_files = [c + '_combined.h5ad' for c in cell_types]
        data = [ad.read_h5ad(in_path / f'{i}') for i in input_files]
        adata_combined = ad.concat(data, join='outer')
        adata_combined.write_h5ad(combined_file)
    datasets[f"'_'.join(cell_types)"] = adata_combined
else:
    input = 'test1'
    datasets[input] = ad.read_h5ad(in_path / f'{input}.h5ad')

In [6]:
# get the order of the original genes (for verification after processing)
genes_ordered = datasets[list(datasets.keys())[0]].var_names.tolist()

In [7]:
# create labels for AD, NCI and MCIP status, 
# based on cogdx, braaksc, and ceradc scores
for label, adata in datasets.items():
    datasets[label] = pre.create_new_labels(adata, drop_unknown=True)

Stratifying subjects based on cogdx, braaksc, and ceradsc...
Dropping subjects with 'unknown' label...
Removed 310 subjects. (427 -> 117)


    Stratification summary:
    AD: 42
    NCI: 38
    MCIP: 37
          


In [9]:
# save genes that exist in reactome to file
reactome_genes = pre.save_reactome_genes(pp.pathway_data_path, 
        filename_read='ReactomePathways.gmt', 
        filename_save='reactome_genes.txt'
        )

Saving list of genes that are in Reactome
11939 Reactome genes written to D:\BBTX11-VT26-07\simulated\data\PathwayData\reactome_genes.txt


In [10]:
# to inspect final objects
final_datasets = {}

for label, adata in datasets.items():
    # keep only genes that are in Reactome
    adata = pre.filter_reactome_genes(adata, set(reactome_genes))

    ## Quality Control
    # remove cells with less than n expressed genes
    adata = pre.filter_cells_by_min_genes(adata, min_genes=min_genes)
    
    # filter genes to keep only genes expressed in at least n cells
    adata = pre.filter_genes_by_min_expression(adata, min_cells=min_cells)

    # remove cells with mito gene expression over threshold
    adata = pre.filter_cells_by_mitochondrial_content(adata, mt_threshold=mito_threshold)

    # filter genes to keep only subjects with at least n cells
    adata = pre.filter_low_cell_subjects(adata, min_cells_per_subject=min_cells_per_subject)

    datasets[label] = adata
    
    # Stratify AD+NCI, MCIP
    adata, mcip_adata = pre.separate_stratified_data(
            adata,
            save_dir=pp.stratified_full_pipe,
            file_prefix=[f"{label}_AD_NCI", f"{label}_MCIP"]
        )

    # Split AD+NCI into train/test
    train_subjects, test_subjects = pre.train_test_split_subjects(
            adata,
            test_size=test_size,
            random_state=random_state,
            stratify_key=stratify_key,
            save_dir=pp.stratified_path,
            file_prefix=label
        )

    # Split the AnnData object into training and testing sets based on the provided subject lists
    train_adata, test_adata = pre.train_test_split_adata(
        adata,
        train_subjects=train_subjects,
        test_subjects=test_subjects,
        save_dir=pp.stratified_path,
        file_prefix=label
    )

    # Save the training, testing and MCIP sets to disk as anndata files
    sc.write(pp.compl_hvg_from_cv / f'{label}_train_preprocessed_noHVG.h5ad', 
             train_adata)
    sc.write(pp.compl_hvg_from_cv / f'{label}_test_preprocessed_noHVG.h5ad', 
             test_adata)
    sc.write(pp.compl_hvg_from_cv / f'{label}_mcip_preprocessed_noHVG.h5ad', 
             mcip_adata)

    final_datasets[f'train_{label}'] = train_adata
    final_datasets[f'test_{label}'] = test_adata
    final_datasets[f'mcip_{label}'] = mcip_adata

Filtering AnnData object to only include genes present in the Reactome database...
Removed 6755 genes. (16121 -> 9366)

Filtering cells with <200 genes...
Filtered data, shape=(2720, 9366)
2720 -> 2720 cells (removed 0)

Filtering genes expressed in <200 cells...
Removed 73 genes. (9366 -> 9293)

Filtering cells with > 5.0% mitochondrial genes...
Removed 0 cells. (2720 -> 2720)

Filtering subjects with < 30 cells...
Removed 106 subjects. (117 -> 11)

Separating AnnData object into stratified datasets...
AD+NCI dataset: 259 cells from 8 subjects
MCIP dataset: 96 cells from 3 subjects

Saved AD+NCI dataset to D:\BBTX11-VT26-07\simulated\data\processed_data\stratified\full_pipeline\test1_AD_NCI_adata.h5ad
Saved MCIP dataset to D:\BBTX11-VT26-07\simulated\data\processed_data\stratified\full_pipeline\test1_MCIP_adata.h5ad

Saved AD+NCI metadata to D:\BBTX11-VT26-07\simulated\data\processed_data\stratified\full_pipeline\test1_AD_NCI_metadata.csv
Saved MCIP metadata to D:\BBTX11-VT26-07\simul

In [11]:
# verify genes in correct order
verify = {'train': train_adata, 'test': test_adata, 'mcip':mcip_adata}
pre.verify_gene_order(verify, genes_ordered)

SUCCESS: All datasets maintain original relative gene order.


In [12]:
for label, adata in final_datasets.items():
    print(adata)


AnnData object with n_obs × n_vars = 195 × 9293
    obs: 'subject', 'msex', 'age_death', 'pmi', 'race', 'Pathologic_diagnosis_of_AD', 'cogdx', 'braaksc', 'ceradsc', 'label', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito'
    var: 'n_cells', 'mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    layers: None (.X)
AnnData object with n_obs × n_vars = 64 × 9293
    obs: 'subject', 'msex', 'age_death', 'pmi', 'race', 'Pathologic_diagnosis_of_AD', 'cogdx', 'braaksc', 'ceradsc', 'label', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_coun